In [1]:
import numpy as np
import torch
import plotly.graph_objects as go

In [46]:
batch_size = 1
hidden_dim = 4096
k = 12

torch.manual_seed(42)
np.random.seed(42)

# Hidden states: [batch_size, hidden_dim] ~ N(0, 1)
hidden_state = torch.randn(batch_size, hidden_dim)

print(f"hidden_state shape: {hidden_state.shape}")
print(f"mean: {hidden_state.mean():.4f}")
print(f"std: {hidden_state.std():.4f}")

hidden_state shape: torch.Size([1, 4096])
mean: -0.0011
std: 0.9961


In [3]:
# k=12 equidistant points on the unit sphere (icosahedron vertices)
phi = (1.0 + np.sqrt(5.0)) / 2.0

raw = np.array([
    [ 0,  1,  phi], [ 0,  1, -phi],
    [ 0, -1,  phi], [ 0, -1, -phi],
    [ 1,  phi,  0], [ 1, -phi,  0],
    [-1,  phi,  0], [-1, -phi,  0],
    [ phi,  0,  1], [ phi,  0, -1],
    [-phi,  0,  1], [-phi,  0, -1],
], dtype=float)

# Normalise to unit sphere
sphere_points = raw / np.linalg.norm(raw[0])   # all rows have |·|=1 by symmetry

print(f"sphere_points shape : {sphere_points.shape}")
print(f"radii (should all be 1): {np.linalg.norm(sphere_points, axis=1).round(6)}")

dists = np.linalg.norm(sphere_points[:, None] - sphere_points[None, :], axis=-1)
np.fill_diagonal(dists, np.inf)
nn_dist = dists.min(axis=1)
print(f"nearest-neighbour distances: {np.unique(nn_dist.round(6))}")

sphere_points shape : (12, 3)
radii (should all be 1): [1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]
nearest-neighbour distances: [1.051462]


In [4]:
random_dim_indices = np.random.choice(hidden_dim, size=3, replace=False)
print(f"random_dim_indices: {random_dim_indices}")

selected = hidden_state[0, random_dim_indices].numpy()
print(f"selected values: {selected.round(4)}")

random_dim_indices: [1299 1338 3873]
selected values: [ 0.9852 -0.0522  1.9953]


In [5]:
projected_3d = selected
norm = np.linalg.norm(projected_3d)
projected_unit = projected_3d / norm

print(f"Raw 3-D point: {projected_3d.round(4)}")
print(f"Unit vector: {projected_unit.round(4)}")
print(f"radius (=1?): {np.linalg.norm(projected_unit):.6f}")

Raw 3-D point: [ 0.9852 -0.0522  1.9953]
Unit vector: [ 0.4426 -0.0234  0.8964]
radius (=1?): 1.000000


In [7]:
# Find closest k-point
similarities  = sphere_points @ projected_unit
closest_index = int(np.argmax(similarities))

arc_dist = np.degrees(np.arccos(np.clip(similarities[closest_index], -1, 1)))
print(f"dims picked: {random_dim_indices}")
print(f"projected onto: {projected_unit.round(4)}")
print(f"closest k-point: #{closest_index}  {sphere_points[closest_index].round(4)}")
print(f"arc distance: {arc_dist:.2f}°")

dims picked: [1299 1338 3873]
projected onto: [ 0.4426 -0.0234  0.8964]
closest k-point: #8  [0.8507 0.     0.5257]
arc distance: 32.03°


In [39]:
POINT_COLOR   = '#e74c3c'
KPOINT_COLOR  = '#3498db'

traces = []

# unit sphere surface
u = np.linspace(0, 2 * np.pi, 60)
v = np.linspace(0, np.pi, 40)
traces.append(go.Surface(
    x=np.outer(np.cos(u), np.sin(v)),
    y=np.outer(np.sin(u), np.sin(v)),
    z=np.outer(np.ones(len(u)), np.cos(v)),
    opacity=0.06,
    colorscale=[[0, 'steelblue'], [1, 'steelblue']],
    showscale=False,
    hoverinfo='skip',
    name='unit sphere',
    contours=dict(
        x=dict(highlight=False),
        y=dict(highlight=False),
        z=dict(highlight=False),
    ),
))

# k-points
labels = [
        f'k{i}<br>({sphere_points[i,0]:.2f}, {sphere_points[i,1]:.2f}, {sphere_points[i,2]:.2f})'
        for i in range(k)
    ]
traces.append(go.Scatter3d(
    x=sphere_points[:, 0], y=sphere_points[:, 1], z=sphere_points[:, 2],
    mode='markers+text',
    marker=dict(size=6, color=KPOINT_COLOR, opacity=0.9),
    text=[f'k{i}' for i in range(k)],
    textposition='top center',
    textfont=dict(size=9, color=KPOINT_COLOR),
    hovertext=labels,
    hoverinfo='text',
    name='k-points',
))

# projected point
pt = projected_unit
closest = sphere_points[closest_index]
traces.append(go.Scatter3d(
    x=[pt[0]], y=[pt[1]], z=[pt[2]],
    mode='markers',
    marker=dict(size=10, color=POINT_COLOR, symbol='diamond', opacity=1.0),
    hovertext=[f'projected<br>dims: {random_dim_indices}<br>'
               f'({pt[0]:.3f}, {pt[1]:.3f}, {pt[2]:.3f})'],
    hoverinfo='text',
    name=f'dims {random_dim_indices} → k{closest_index} ({arc_dist:.1f}°)',
))

# line: projected → closest k-point
traces.append(go.Scatter3d(
    x=[pt[0], closest[0]], y=[pt[1], closest[1]], z=[pt[2], closest[2]],
    mode='lines',
    line=dict(color=POINT_COLOR, width=4),
    showlegend=False,
    hoverinfo='skip',
))

# arrowhead cone at the closest k-point
traces.append(go.Cone(
    x=[closest[0]], y=[closest[1]], z=[closest[2]],
    u=[closest[0] - pt[0]], v=[closest[1] - pt[1]], w=[closest[2] - pt[2]],
    sizemode='absolute', sizeref=0.08,
    colorscale=[[0, POINT_COLOR], [1, POINT_COLOR]],
    showscale=False,
    hoverinfo='skip',
    anchor='tip',
))

fig = go.Figure(traces)
fig.update_layout(
    title=dict(
        text=(f'Sphere projection  |  batch_size={batch_size}, hidden_dim={hidden_dim}<br>'
              f'<sup>k=12 vertices · 3 dims picked → normalised → matched</sup>'),
        x=0.5,
    ),
    legend=dict(x=0.01, y=0.99),
    width=800, height=750,
)
fig.show()

In [49]:
from plotly.subplots import make_subplots

DATA_PATH = "results/hidden_states_Qwen2.5-7B-stat-256.npz"

raw = np.load(DATA_PATH, allow_pickle=True)
block_hashes_all = raw["block_hashes"]
nonces_all = raw["nonces"]
hidden_states_all = raw["hidden_states"]
reduced_hidden_states_all = raw["reduced_hidden_states"]

N, D = hidden_states_all.shape

unique_hashes = list(dict.fromkeys(block_hashes_all.tolist()))
print(f"Records: {N}")
print(f"Hidden dim: {D}")
print(f"Unique hashes: {len(unique_hashes)}")

Records: 1024
Hidden dim: 3584
Unique hashes: 4


In [50]:
PALETTE = [
    '#e74c3c', '#2ecc71', '#3498db', '#f39c12',
    '#9b59b6', '#1abc9c', '#e67e22', '#34495e',
]

# Compute per-hash mean hidden state (mean over nonces)
hash_means = {}
hash_nonces = {}
for h in unique_hashes:
    mask = block_hashes_all == h
    hash_means[h] = hidden_states_all[mask].mean(axis=0)
    hash_nonces[h] = nonces_all[mask].tolist()

fig = go.Figure()

for h, color in zip(unique_hashes, PALETTE):
    mean_vec = hash_means[h]
    n_avg = len(hash_nonces[h])
    label = f"{h[:14]}…  (avg of {n_avg} nonce{'s' if n_avg > 1 else ''})"

    fig.add_trace(go.Histogram(
        x=mean_vec,
        name=label,
        nbinsx=120,
        opacity=0.65,
        marker_color=color,
    ))

fig.update_layout(
    barmode="overlay",
    title=dict(
        text=(f"Mean hidden-state value distribution per hash  |  D={D}<br>"
              f"<sup>Each trace = mean over all nonces within that hash.</sup>"),
        x=0.5,
    ),
    xaxis_title="Value",
    yaxis_title="Count",
    legend=dict(x=0.01, y=0.99),
    width=880,
    height=480,
)
fig.show()

In [ ]:
norms = np.linalg.norm(reduced_hidden_states_all, axis=1, keepdims=True)
proj  = reduced_hidden_states_all / (norms + 1e-8)

sims = proj @ sphere_points.T
closest_idxs = np.argmax(sims, axis=1)
arc_dists = np.degrees(
    np.arccos(np.clip(sims[np.arange(N), closest_idxs], -1, 1))
)

# for i in range(N):
#     print(f"hash={block_hashes_all[i][:12]}…  nonce={nonces_all[i]:4d}"
#           f"proj={proj[i].round(3)}  → k{closest_idxs[i]}  arc={arc_dists[i]:.2f}°")

KPOINT_COLOR = 'black'
nn_thr = dists.min() * 1.01 # edge threshold

traces = []

u_s = np.linspace(0, 2 * np.pi, 60)
v_s = np.linspace(0, np.pi, 40)
traces.append(go.Surface(
    x=np.outer(np.cos(u_s), np.sin(v_s)),
    y=np.outer(np.sin(u_s), np.sin(v_s)),
    z=np.outer(np.ones(len(u_s)), np.cos(v_s)),
    opacity=0.1,
    colorscale=[[0, 'steelblue'], [1, 'steelblue']],
    showscale=False, hoverinfo='skip',
    contours=dict(
        x=dict(highlight=False),
        y=dict(highlight=False),
        z=dict(highlight=False),
    ),
))

# k-points
traces.append(go.Scatter3d(
    x=sphere_points[:, 0], y=sphere_points[:, 1], z=sphere_points[:, 2],
    mode='markers+text',
    marker=dict(size=6, color=KPOINT_COLOR, opacity=0.9),
    text=[f'k{i}' for i in range(k)],
    textposition='top center',
    textfont=dict(size=9, color=KPOINT_COLOR),
    hovertext=[
        f'k{i}<br>({sphere_points[i,0]:.2f}, {sphere_points[i,1]:.2f}, {sphere_points[i,2]:.2f})'
        for i in range(k)
    ],
    hoverinfo='text',
    name='k-points',
))

# projected points grouped by hash
for hi, (h, color) in enumerate(zip(unique_hashes, PALETTE)):
    mask   = np.where(block_hashes_all == h)[0]
    pts    = proj[mask]           # [n, 3]
    ns     = nonces_all[mask]
    ci     = closest_idxs[mask]
    arc_d  = arc_dists[mask]

    # scatter points
    traces.append(go.Scatter3d(
        x=pts[:, 0], y=pts[:, 1], z=pts[:, 2],
        mode='markers',
        marker=dict(size=10, color=color, symbol='diamond', opacity=1.0),
        hovertext=[
            f"hash: {h[:14]}…<br>nonce: {ns[j]}<br>"
            f"proj: ({pts[j,0]:.3f}, {pts[j,1]:.3f}, {pts[j,2]:.3f})<br>"
            f"→ k{ci[j]}  arc={arc_d[j]:.2f}°"
            for j in range(len(mask))
        ],
        hoverinfo='text',
        name=f"{h[:14]}…",
        legendgroup=h,
    ))

    # arrow lines to closest k-point
    for j in range(len(mask)):
        pt      = pts[j]
        closest = sphere_points[ci[j]]
        traces.append(go.Scatter3d(
            x=[pt[0], closest[0]], y=[pt[1], closest[1]], z=[pt[2], closest[2]],
            mode='lines', line=dict(color=color, width=3),
            showlegend=False, hoverinfo='skip', legendgroup=h,
        ))
        traces.append(go.Cone(
            x=[closest[0]], y=[closest[1]], z=[closest[2]],
            u=[closest[0] - pt[0]], v=[closest[1] - pt[1]], w=[closest[2] - pt[2]],
            sizemode='absolute', sizeref=0.07,
            colorscale=[[0, color], [1, color]],
            showscale=False, hoverinfo='skip', anchor='tip',
        ))

fig = go.Figure(traces)
fig.update_layout(
    title=dict(
        text=(f"Reduced hidden states on unit sphere  |  N={N}, hashes={len(unique_hashes)}<br>"
              f"<sup>Each point = one (hash, nonce). Arrow → nearest k-point.</sup>"),
        x=0.5,
    ),
    width=850, height=780,
)
fig.show()
# fig.write_html("sphere_projection.html")

In [52]:
import collections

overall_counts = np.bincount(closest_idxs, minlength=k)

hash_counts = {}
for h in unique_hashes:
    mask = block_hashes_all == h
    hash_counts[h] = np.bincount(closest_idxs[mask], minlength=k)

k_labels = [f"k{i}" for i in range(k)]

fig = go.Figure()

for h, color in zip(unique_hashes, PALETTE):
    counts = hash_counts[h]
    fig.add_trace(go.Bar(
        x=k_labels,
        y=counts,
        name=f"{h[:14]}…",
        marker_color=color,
        opacity=0.85,
        hovertemplate="k-point: %{x}<br>count: %{y}<extra>%{fullData.name}</extra>",
    ))

fig.add_trace(go.Scatter(
    x=k_labels,
    y=overall_counts,
    mode="markers+text",
    name="total",
    line=dict(color="black", width=2, dash="dot"),
    marker=dict(size=7, color="black"),
    text=[str(c) if c > 0 else "" for c in overall_counts],
    textposition="top center",
    textfont=dict(size=10),
))

fig.update_layout(
    barmode="stack",
    title=dict(
        text=(f"Nearest k-point assignment  |  N={N}, hashes={len(unique_hashes)}<br>"
              f"<sup>How many projected vectors were closest to each vertex.</sup>"),
        x=0.5,
    ),
    xaxis=dict(title="k-point", categoryorder="array",
               categoryarray=k_labels),
    yaxis=dict(title="Number of samples assigned"),
    width=820,
    height=460,
)
fig.show()